In [ ]:
import polars as pl
import numpy as np
import os
import gc

output_dir = "/kaggle/working/"
train_dir = "/kaggle/working/processed_train/"
test_dir = "/kaggle/working/processed_test/"
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# 🌟 1. 读取已经聚合成用户序列的 Parquet，并进行用户级别切分 (90% Train, 10% Test)
# 原始 13.3 亿行 Tmall 日志请先通过流式排序/聚合生成 sequence parquet。
# ⚠️ 注意：请确保下方这里的路径是你真实的单文件路径
SINGLE_FILE_PATH = "/kaggle/input/datasets/kekekem/tmall-recommendation-data/chunks/hstu_seq_chunk_00.parquet" 
print(f"📦 正在读取单文件: {SINGLE_FILE_PATH}")
# Memory-safe default: only read a bounded number of already-aggregated users.
# Set to None only after the small run is stable and the runtime has enough RAM.
MAX_SOURCE_USERS = 50_000
source_scan = pl.scan_parquet(SINGLE_FILE_PATH).select([
    'user_id', 'item_seq', 'brand_seq', 'cat_large_seq',
    'behavior_seq', 'time_gap_seq',
])
if MAX_SOURCE_USERS is not None:
    source_scan = source_scan.head(MAX_SOURCE_USERS)
df_all = source_scan.collect(streaming=True)
del source_scan

# 不再复制整张表做全量 shuffle，按文件中的用户顺序切分以节省内存。
split_idx = int(0.9 * len(df_all))

train_df = df_all.head(split_idx)
test_df = df_all.tail(len(df_all) - split_idx)
print(f"📅 用户切分完成: 训练集 {len(train_df)} 个用户, 测试集 {len(test_df)} 个用户")

# 🌟 2. 提取全局词表 (修复：改为直接处理 DataFrame，仅使用训练集)
def build_map_from_df(df, col):
    print(f"统计 {col} 全局词表...")
    # 从 DataFrame 中提取唯一 ID
    ids = df.select(col).explode(col).drop_nulls().unique()[col].to_list()
    sorted_ids = np.array(sorted(ids), dtype=np.int64)
    return pl.DataFrame({col: sorted_ids, f"{col}_idx": np.arange(1, len(sorted_ids) + 1, dtype=np.int32)})

item_map = build_map_from_df(train_df, "item_seq")
brand_map = build_map_from_df(train_df, "brand_seq")
cat_map = build_map_from_df(train_df, "cat_large_seq")
beh_map = build_map_from_df(train_df, "behavior_seq")

np.save(f"{output_dir}sorted_item_ids.npy", item_map["item_seq"].to_numpy())
np.save(f"{output_dir}sorted_brand_ids.npy", brand_map["brand_seq"].to_numpy()) 
np.save(f"{output_dir}sorted_cat_ids.npy", cat_map["cat_large_seq"].to_numpy()) 
np.save(f"{output_dir}sorted_beh_ids.npy", beh_map["behavior_seq"].to_numpy()) 

item_sorted_arr = item_map["item_seq"].to_numpy()
brand_sorted_arr = brand_map["brand_seq"].to_numpy()
cat_sorted_arr = cat_map["cat_large_seq"].to_numpy()
beh_sorted_arr = beh_map["behavior_seq"].to_numpy()


# 🌟 3. Legacy ItemCF baseline
# 本版本不在主预处理链路中重复构建原始 ID ItemCF 表。
# 协同共现、时间/行为加权和邻居截断统一在下一单元基于 remapped ID 完成。

# 🌟 4. 全量映射数据 (将拆分好的 DataFrame 保存)
def extract_and_map(df, col_name, sorted_arr):
    """辅助函数：安全地重映射 ID，将未登录词(OOV)映射为 0"""
    lens = df[col_name].list.len().to_numpy()
    offsets = np.zeros(len(lens) + 1, dtype=np.int32)
    offsets[1:] = np.cumsum(lens)
    
    flat_vals = df[col_name].explode().to_numpy()
    
    idx = np.searchsorted(sorted_arr, flat_vals)
    idx_safe = np.clip(idx, 0, len(sorted_arr) - 1)
    valid_mask = sorted_arr[idx_safe] == flat_vals
    mapped_flat = np.where(valid_mask, idx_safe + 1, 0)
    
    return [mapped_flat[offsets[j]:offsets[j+1]].tolist() for j in range(len(offsets)-1)]

def process_and_save_df(df, save_path, prefix=""):
    print(f"🚀 开始映射并保存 {prefix} 数据...")
    mapped_item = extract_and_map(df, "item_seq", item_sorted_arr)
    mapped_brand = extract_and_map(df, "brand_seq", brand_sorted_arr)
    mapped_cat = extract_and_map(df, "cat_large_seq", cat_sorted_arr)
    mapped_beh = extract_and_map(df, "behavior_seq", beh_sorted_arr)
    
    df_mapped = df.with_columns([
        pl.Series("item_seq", mapped_item),
        pl.Series("brand_seq", mapped_brand),
        pl.Series("cat_large_seq", mapped_cat),
        pl.Series("behavior_seq", mapped_beh)
    ])
    
    df_mapped.write_parquet(save_path)
    del df_mapped
    gc.collect()

# 这里不再需要循环了，直接对刚才切分出来的两份数据做处理
process_and_save_df(train_df, f"{train_dir}chunk_0.parquet", "Train")
process_and_save_df(test_df, f"{test_dir}chunk_0.parquet", "Test")

# 🌟 5. 特征严格隔离 B：生成全局热度 Product Info - 【仅限训练集】
print("🔄 正在生成 Product Info (严格无未来泄露)...")
# 不用从本地重读文件，直接基于内存中的 train_df 生成
product_info = (
    train_df.select(["item_seq", "cat_large_seq"])
    .explode(["item_seq", "cat_large_seq"])
    .group_by("item_seq")
    .agg([
        pl.col("cat_large_seq").last().alias("cat_large"),
        pl.len().alias("item_hotness")
    ])
)
product_info.write_parquet(f"{output_dir}item_product_info.parquet")
print("✨ 全部去偏与 OOT 预处理完成！真实的离线环境已就绪。")

In [ ]:
import gc
import math
from collections import defaultdict

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


# ================================================================
# 1. Collaborative signal: weighted item-item co-occurrence
# ================================================================
# This cell never materializes an N_item x N_item dense matrix.  It creates a
# bounded sparse pair table from local behavior windows, then learns dense item
# embeddings with a weighted Item2Vec / negative-sampling objective.
TRAIN_SEQUENCE_FILE = '/kaggle/working/processed_train/chunk_0.parquet'
PAIR_FILE = '/kaggle/working/collaborative_item_pairs.parquet'
ITEM2VEC_FILE = '/kaggle/working/collaborative_item2vec.pt'
RQ_VAE_FILE = '/kaggle/working/rqvae_tokenizer.pt'
SEMANTIC_ID_FILE = '/kaggle/working/rqvae_semantic_id_table.parquet'
SEMANTIC_ID_ARRAY = '/kaggle/working/rqvae_semantic_id_table.npy'

COOC_WINDOW = 2
MAX_COOC_USERS = 10_000
MAX_USER_EVENTS = 80
MAX_PAIR_ROWS_PER_PART = 200_000
MAX_NEIGHBORS_PER_ITEM = 50
MAX_TRAIN_PAIRS = 300_000
MAX_MODEL_ITEMS = 500_000
TIME_DECAY_SCALE = 20.0

# processed behavior IDs are normally 1=click, 2=collect, 3=cart, 4=alipay.
# Verify sorted_beh_ids.npy if the upstream mapping is changed.
BEHAVIOR_WEIGHT_BY_ID = {1: 1.0, 2: 3.0, 3: 5.0, 4: 8.0}

sequence_train_df = (
    pl.scan_parquet(TRAIN_SEQUENCE_FILE)
    .select(['user_id', 'item_seq', 'behavior_seq', 'time_gap_seq'])
    .head(MAX_COOC_USERS)
    .collect(streaming=True)
)
required_cols = {'user_id', 'item_seq', 'behavior_seq', 'time_gap_seq'}
missing_cols = required_cols.difference(sequence_train_df.columns)
if missing_cols:
    raise ValueError(f'Missing required sequence columns: {sorted(missing_cols)}')

sequence_train_df = sequence_train_df.with_columns([
    pl.col('item_seq').list.tail(MAX_USER_EVENTS),
    pl.col('behavior_seq').list.tail(MAX_USER_EVENTS),
    pl.col('time_gap_seq').list.tail(MAX_USER_EVENTS),
])
events = (
    sequence_train_df
    .select(['user_id', 'item_seq', 'behavior_seq', 'time_gap_seq'])
    .with_columns(pl.col('item_seq').list.len().alias('user_seq_len'))
    .explode(['item_seq', 'behavior_seq', 'time_gap_seq'])
    .filter(pl.col('item_seq') > 0)
    .with_columns(
        (1.0 / pl.col('user_seq_len').cast(pl.Float64).log1p()).alias('user_penalty')
    )
)


def behavior_weight_expr(column_name):
    expr = pl.when(pl.col(column_name) == 4).then(pl.lit(8.0))
    expr = expr.when(pl.col(column_name) == 3).then(pl.lit(5.0))
    expr = expr.when(pl.col(column_name) == 2).then(pl.lit(3.0))
    return expr.otherwise(pl.lit(1.0))


pair_parts = []
for offset in range(1, COOC_WINDOW + 1):
    forward = (
        events
        .with_columns([
            pl.col('item_seq').shift(-offset).over('user_id').alias('context_item'),
            pl.col('behavior_seq').shift(-offset).over('user_id').alias('context_behavior'),
            pl.col('time_gap_seq').shift(-offset).over('user_id').alias('context_gap'),
        ])
        .drop_nulls(['context_item', 'context_behavior', 'context_gap'])
        .filter(pl.col('item_seq') != pl.col('context_item'))
        .with_columns([
            behavior_weight_expr('context_behavior').alias('behavior_weight'),
            (-pl.col('context_gap').abs().cast(pl.Float64) / TIME_DECAY_SCALE)
            .exp().alias('time_weight'),
        ])
        .with_columns(
            (
                pl.col('behavior_weight')
                * pl.col('time_weight')
                * pl.col('user_penalty')
                * (0.8 ** (offset - 1))
            ).alias('pair_weight')
        )
        .select([
            pl.col('item_seq').cast(pl.Int32).alias('src_item'),
            pl.col('context_item').cast(pl.Int32).alias('dst_item'),
            pl.col('pair_weight').cast(pl.Float32),
        ])
    )
    if forward.height > MAX_PAIR_ROWS_PER_PART:
        forward = forward.sample(n=MAX_PAIR_ROWS_PER_PART, seed=offset)
    reverse = forward.select([
        pl.col('dst_item').alias('src_item'),
        pl.col('src_item').alias('dst_item'),
        pl.col('pair_weight'),
    ])
    pair_parts.extend([forward, reverse])

pair_df = (
    pl.concat(pair_parts)
    .group_by(['src_item', 'dst_item'])
    .agg(pl.col('pair_weight').sum())
    .sort(['src_item', 'pair_weight'], descending=[False, True])
    .group_by('src_item', maintain_order=True)
    .head(MAX_NEIGHBORS_PER_ITEM)
)

if len(pair_df) > MAX_TRAIN_PAIRS:
    pair_df = pair_df.sample(n=MAX_TRAIN_PAIRS, seed=42)
pair_df.write_parquet(PAIR_FILE)

max_item_index = int(events.select(pl.col('item_seq').max()).item())
num_items = min(max_item_index + 1, MAX_MODEL_ITEMS + 1)  # index 0 is padding
pair_df = pair_df.filter(
    (pl.col('src_item') < num_items) & (pl.col('dst_item') < num_items)
)
hot_item_pool = (
    events.group_by('item_seq').len()
    .sort('len', descending=True)
    .filter(pl.col('item_seq') < num_items)
    .head(min(50_000, max(0, num_items - 1)))['item_seq']
    .to_numpy()
)

print(f'Collaborative pairs: {len(pair_df):,}')
print(f'Active item vocabulary used by model: {num_items - 1:,}')
del events
gc.collect()


# ================================================================
# 2. Weighted Item2Vec / contrastive item embedding
# ================================================================
class ItemPairDataset(Dataset):
    def __init__(self, pair_frame):
        self.src = pair_frame['src_item'].to_numpy().astype(np.int64)
        self.dst = pair_frame['dst_item'].to_numpy().astype(np.int64)
        weights = np.log1p(pair_frame['pair_weight'].to_numpy().astype(np.float32))
        self.weight = weights / max(float(weights.mean()), 1e-6)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, index):
        return (
            torch.tensor(self.src[index], dtype=torch.long),
            torch.tensor(self.dst[index], dtype=torch.long),
            torch.tensor(self.weight[index], dtype=torch.float32),
        )


class WeightedItem2Vec(nn.Module):
    def __init__(self, num_items, embedding_dim=32):
        super().__init__()
        self.item_emb = nn.Embedding(
            num_items, embedding_dim, padding_idx=0, sparse=True
        )
        nn.init.normal_(self.item_emb.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.item_emb.weight[0].zero_()

    def forward(self, src_item, pos_item, neg_item, pair_weight):
        src = F.normalize(self.item_emb(src_item), p=2, dim=-1)
        pos = F.normalize(self.item_emb(pos_item), p=2, dim=-1)
        neg = F.normalize(self.item_emb(neg_item), p=2, dim=-1)

        pos_score = (src * pos).sum(dim=-1)
        neg_score = torch.einsum('bd,bnd->bn', src, neg)
        pos_loss = -F.logsigmoid(pos_score)
        neg_loss = -F.logsigmoid(-neg_score).mean(dim=1)
        return (pair_weight * (pos_loss + neg_loss)).mean()


ITEM_EMBED_DIM = 16
ITEM2VEC_BATCH_SIZE = 1024
ITEM2VEC_EPOCHS = 1
NUM_NEGATIVES = 8

pair_dataset = ItemPairDataset(pair_df)
del pair_df
gc.collect()
pair_loader = DataLoader(
    pair_dataset, batch_size=ITEM2VEC_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=False,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
item2vec = WeightedItem2Vec(num_items, ITEM_EMBED_DIM).to(device)
item2vec_optimizer = torch.optim.SparseAdam(item2vec.parameters(), lr=2e-3)
hot_item_pool_tensor = torch.as_tensor(hot_item_pool, dtype=torch.long, device=device)

for epoch in range(ITEM2VEC_EPOCHS):
    item2vec.train()
    epoch_losses = []
    for src_item, pos_item, pair_weight in tqdm(
        pair_loader, desc=f'Item2Vec epoch {epoch + 1}'
    ):
        src_item = src_item.to(device, non_blocking=True)
        pos_item = pos_item.to(device, non_blocking=True)
        pair_weight = pair_weight.to(device, non_blocking=True)
        batch_size = src_item.size(0)

        num_uniform = NUM_NEGATIVES // 2
        uniform_neg = torch.randint(
            1, num_items, (batch_size, num_uniform), device=device
        )
        hot_indices = torch.randint(
            0, len(hot_item_pool_tensor),
            (batch_size, NUM_NEGATIVES - num_uniform), device=device,
        )
        hot_neg = hot_item_pool_tensor[hot_indices]
        neg_item = torch.cat([uniform_neg, hot_neg], dim=1)

        item2vec_optimizer.zero_grad(set_to_none=True)
        loss = item2vec(src_item, pos_item, neg_item, pair_weight)
        loss.backward()
        item2vec_optimizer.step()
        epoch_losses.append(float(loss.item()))

    print(f'Item2Vec epoch {epoch + 1} | loss={np.mean(epoch_losses):.4f}')

torch.save(item2vec.state_dict(), ITEM2VEC_FILE)


# ================================================================
# 3. RQ-VAE tokenizer: collaborative embedding -> hierarchical SID
# ================================================================
class RQVAETokenizer(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, latent_dim=32,
                 num_levels=3, codebook_size=256, commitment_beta=0.25):
        super().__init__()
        self.num_levels = num_levels
        self.codebook_size = codebook_size
        self.commitment_beta = commitment_beta
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim),
        )
        self.codebooks = nn.ModuleList([
            nn.Embedding(codebook_size, latent_dim)
            for _ in range(num_levels)
        ])
        for codebook in self.codebooks:
            nn.init.uniform_(codebook.weight, -1.0 / codebook_size, 1.0 / codebook_size)

    def quantize(self, z):
        residual = z
        quantized_sum = torch.zeros_like(z)
        all_codes = []
        for codebook in self.codebooks:
            code_vectors = codebook.weight
            distances = (
                residual.pow(2).sum(dim=1, keepdim=True)
                + code_vectors.pow(2).sum(dim=1).unsqueeze(0)
                - 2.0 * residual @ code_vectors.t()
            )
            code = distances.argmin(dim=1)
            quantized = codebook(code)
            all_codes.append(code)
            quantized_sum = quantized_sum + quantized
            residual = residual - quantized
        return torch.stack(all_codes, dim=1), quantized_sum

    def forward(self, item_embedding):
        z = self.encoder(item_embedding)
        codes, quantized = self.quantize(z)
        straight_through = z + (quantized - z).detach()
        reconstruction = self.decoder(straight_through)

        recon_loss = F.mse_loss(reconstruction, item_embedding)
        codebook_loss = F.mse_loss(quantized, z.detach())
        commitment_loss = F.mse_loss(z, quantized.detach())
        loss = recon_loss + codebook_loss + self.commitment_beta * commitment_loss
        return reconstruction, codes, loss, {
            'recon': recon_loss.detach(),
            'codebook': codebook_loss.detach(),
            'commitment': commitment_loss.detach(),
        }

    @torch.no_grad()
    def encode_codes(self, item_embedding):
        z = self.encoder(item_embedding)
        codes, _ = self.quantize(z)
        return codes


class ItemIndexDataset(Dataset):
    def __init__(self, max_item_index):
        self.item_ids = np.arange(1, max_item_index + 1, dtype=np.int64)

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, index):
        return torch.tensor(self.item_ids[index], dtype=torch.long)


RQ_LEVELS = 3
RQ_CODEBOOK_SIZE = 256
RQ_BATCH_SIZE = 1024
RQ_EPOCHS = 2

item_index_dataset = ItemIndexDataset(num_items - 1)
item_index_loader = DataLoader(
    item_index_dataset, batch_size=RQ_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False,
)
rqvae = RQVAETokenizer(
    input_dim=ITEM_EMBED_DIM,
    hidden_dim=64,
    latent_dim=16,
    num_levels=RQ_LEVELS,
    codebook_size=RQ_CODEBOOK_SIZE,
).to(device)
rqvae_optimizer = torch.optim.AdamW(rqvae.parameters(), lr=1e-3, weight_decay=1e-5)

item2vec.eval()
for epoch in range(RQ_EPOCHS):
    rqvae.train()
    epoch_losses = []
    for item_ids in tqdm(item_index_loader, desc=f'RQ-VAE epoch {epoch + 1}'):
        item_ids = item_ids.to(device, non_blocking=True)
        with torch.no_grad():
            item_embedding = F.normalize(item2vec.item_emb(item_ids), p=2, dim=-1)
        rqvae_optimizer.zero_grad(set_to_none=True)
        _, _, loss, _ = rqvae(item_embedding)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rqvae.parameters(), max_norm=5.0)
        rqvae_optimizer.step()
        epoch_losses.append(float(loss.item()))
    print(f'RQ-VAE epoch {epoch + 1} | loss={np.mean(epoch_losses):.6f}')

torch.save(rqvae.state_dict(), RQ_VAE_FILE)


# ================================================================
# 4. Generate SIDs and append a collision token for uniqueness
# ================================================================
all_codes = np.zeros((num_items, RQ_LEVELS), dtype=np.int32)
ordered_item_loader = DataLoader(
    item_index_dataset, batch_size=RQ_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False,
)
rqvae.eval()
with torch.no_grad():
    for item_ids in tqdm(ordered_item_loader, desc='Generating RQ-VAE codes'):
        item_ids_device = item_ids.to(device, non_blocking=True)
        item_embedding = F.normalize(item2vec.item_emb(item_ids_device), p=2, dim=-1)
        # +1 reserves token 0 for padding in the HSTU model.
        codes = rqvae.encode_codes(item_embedding).cpu().numpy().astype(np.int32) + 1
        all_codes[item_ids.numpy()] = codes

sid_frame = pl.DataFrame({
    'item_seq': np.arange(1, num_items, dtype=np.int32),
    'sid_0': all_codes[1:, 0],
    'sid_1': all_codes[1:, 1],
    'sid_2': all_codes[1:, 2],
}).sort(['sid_0', 'sid_1', 'sid_2', 'item_seq'])

sid_frame = sid_frame.with_columns(
    (
        pl.int_range(0, pl.len()).over(['sid_0', 'sid_1', 'sid_2']) + 1
    ).cast(pl.Int32).alias('sid_collision')
)
sid_frame.write_parquet(SEMANTIC_ID_FILE)

sid_rows = sid_frame.select([
    'item_seq', 'sid_0', 'sid_1', 'sid_2', 'sid_collision'
]).to_numpy()
sid_table = np.zeros((num_items, 4), dtype=np.int32)
sid_table[sid_rows[:, 0].astype(np.int64)] = sid_rows[:, 1:].astype(np.int32)
np.save(SEMANTIC_ID_ARRAY, sid_table)

sid_vocab_sizes = [RQ_CODEBOOK_SIZE + 1] * RQ_LEVELS + [
    int(sid_rows[:, 4].max()) + 1
]
sid_to_items = defaultdict(list)
sid_prefix_next = defaultdict(set)
for item_id, code_0, code_1, code_2, collision in sid_rows.tolist():
    sid = (int(code_0), int(code_1), int(code_2), int(collision))
    sid_to_items[sid].append(int(item_id))
    for level, code in enumerate(sid):
        sid_prefix_next[sid[:level]].add(int(code))

# Category is optional and skipped in this low-memory variant.
item_to_cat = {}

collision_rate = 1.0 - (
    sid_frame.select(['sid_0', 'sid_1', 'sid_2']).unique().height
    / max(1, sid_frame.height)
)
print(f'RQ-VAE SID vocab sizes: {sid_vocab_sizes}')
print(f'Pre-disambiguation collision rate: {collision_rate:.4%}')
print(f'Unique final SIDs: {len(sid_to_items):,}')

# Release tokenizer training state before the HSTU phase.
item2vec = item2vec.cpu()
rqvae = rqvae.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
del pair_dataset, pair_loader, item_index_loader, ordered_item_loader
gc.collect()

In [ ]:
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ================================================================
# 1. Dataset: user history -> next Semantic ID
# ================================================================
class GenerativeRecDataset(Dataset):
    def __init__(self, parquet_file, sid_table, max_seq_len=200):
        print(f'Loading sequence dataset: {parquet_file}')
        self.df = pd.read_parquet(
            parquet_file,
            columns=['user_id', 'item_seq', 'behavior_seq', 'time_gap_seq'],
        )
        self.sid_table = sid_table
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _pad_1d(values, length):
        out = np.zeros(length, dtype=np.int64)
        n = min(len(values), length)
        if n:
            out[:n] = np.asarray(values[:n], dtype=np.int64)
        return torch.from_numpy(out)

    @staticmethod
    def _pad_2d(values, length, width):
        out = np.zeros((length, width), dtype=np.int64)
        n = min(len(values), length)
        if n:
            out[:n] = np.asarray(values[:n], dtype=np.int64)
        return torch.from_numpy(out)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_items = np.asarray(row['item_seq'], dtype=np.int64)
        raw_behaviors = np.asarray(row['behavior_seq'], dtype=np.int64)
        raw_time_gap = np.clip(np.asarray(row['time_gap_seq'], dtype=np.int64), 0, 100)

        valid = raw_items > 0
        raw_items = raw_items[valid]
        raw_behaviors = raw_behaviors[valid]
        raw_time_gap = raw_time_gap[valid]

        # Keep one extra event for next-event supervision.
        raw_items = raw_items[-(self.max_seq_len + 1):]
        raw_behaviors = raw_behaviors[-(self.max_seq_len + 1):]
        raw_time_gap = raw_time_gap[-(self.max_seq_len + 1):]

        if len(raw_items) < 2:
            raw_items = np.array([0, 0], dtype=np.int64)
            raw_behaviors = np.array([0, 0], dtype=np.int64)
            raw_time_gap = np.array([0, 0], dtype=np.int64)

        sid_seq = np.zeros((len(raw_items), self.sid_table.shape[1]), dtype=np.int64)
        in_vocab = (raw_items > 0) & (raw_items < len(self.sid_table))
        sid_seq[in_vocab] = self.sid_table[raw_items[in_vocab]]

        input_items = raw_items[:-1]
        target_items = raw_items[1:]
        input_sid = sid_seq[:-1]
        target_sid = sid_seq[1:]
        input_behavior = raw_behaviors[:-1]
        input_time_gap = raw_time_gap[:-1]
        seq_len = min(len(input_items), self.max_seq_len)

        key_padding_mask = torch.cat([
            torch.zeros(seq_len, dtype=torch.bool),
            torch.ones(self.max_seq_len - seq_len, dtype=torch.bool),
        ])

        return {
            'sid': self._pad_2d(input_sid, self.max_seq_len, 4),
            'item': self._pad_1d(input_items, self.max_seq_len),
            'behavior': self._pad_1d(input_behavior, self.max_seq_len),
            'time_gap': self._pad_1d(input_time_gap, self.max_seq_len),
            'target_sid': self._pad_2d(target_sid, self.max_seq_len, 4),
            'target_item': self._pad_1d(target_items, self.max_seq_len),
            'key_padding_mask': key_padding_mask,
        }


# ================================================================
# 2. HSTU backbone + multi-head Semantic-ID generator
# ================================================================
class RelativeAttentionBias(nn.Module):
    def __init__(self, num_heads, num_pos_buckets=32, num_time_buckets=32,
                 max_seq_len=200, max_time=20000):
        super().__init__()
        self.num_heads = num_heads
        self.num_pos_buckets = num_pos_buckets
        self.num_time_buckets = num_time_buckets
        self.max_seq_len = max_seq_len
        self.max_time = max_time
        self.pos_bias = nn.Embedding(num_pos_buckets, num_heads)
        self.time_bias = nn.Embedding(num_time_buckets, num_heads)
        nn.init.zeros_(self.pos_bias.weight)
        nn.init.zeros_(self.time_bias.weight)

    @staticmethod
    def _log_bucket(n, num_buckets, max_val):
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_large = max_exact + (
            torch.log(n.float().clamp(min=1).div(max(max_exact, 1)))
            .div(math.log(max(max_val, max_exact + 1) / max(max_exact, 1)))
            .mul(num_buckets - max_exact)
        ).long().clamp(max=num_buckets - 1)
        return torch.where(is_small, n.clamp(min=0), val_large)

    def forward(self, seq_len, time_gap, device):
        idx = torch.arange(seq_len, device=device)
        rel_pos = (idx.unsqueeze(0) - idx.unsqueeze(1)).clamp(min=0)
        pos_bucket = self._log_bucket(rel_pos, self.num_pos_buckets, self.max_seq_len)
        pos_bias = self.pos_bias(pos_bucket).permute(2, 0, 1).unsqueeze(0)

        cumulative_time = torch.cumsum(time_gap.float(), dim=1)
        delta_time = (
            cumulative_time.unsqueeze(2) - cumulative_time.unsqueeze(1)
        ).clamp(min=0).long()
        time_bucket = self._log_bucket(delta_time, self.num_time_buckets, self.max_time)
        time_bias = self.time_bias(time_bucket).permute(0, 3, 1, 2)
        return pos_bias + time_bias


class HSTUBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_size, eps=1e-12)
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.u_proj = nn.Linear(hidden_size, hidden_size)
        self.o_proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.num_heads = num_heads
        self.d_k = hidden_size // num_heads

    def forward(self, x, causal_mask, key_padding_mask, rel_bias):
        batch_size, seq_len, hidden_size = x.size()
        h = self.ln(x)
        q = self.q_proj(h).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(h).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(h).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        u = F.silu(self.u_proj(h))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores + rel_bias
        scores = scores.masked_fill(causal_mask, -1e4)
        pad_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
        scores = scores.masked_fill(pad_mask, -1e4)
        weights = self.dropout(F.silu(scores))

        context = torch.matmul(weights, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, hidden_size)
        return x + self.dropout(self.o_proj(context * u))


class GenerativeHSTU(nn.Module):
    """HSTU encoder + token-level autoregressive Semantic-ID decoder."""
    def __init__(self, sid_vocab_sizes, behavior_vocab, max_seq_len=200,
                 hidden_size=128, num_heads=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.sid_embeddings = nn.ModuleList([
            nn.Embedding(vocab, hidden_size, padding_idx=0)
            for vocab in sid_vocab_sizes
        ])
        self.behavior_emb = nn.Embedding(behavior_vocab, hidden_size, padding_idx=0)
        self.time_gap_emb = nn.Embedding(101, hidden_size, padding_idx=0)
        self.rel_attn_bias = RelativeAttentionBias(
            num_heads, max_seq_len=max_seq_len
        )
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            HSTUBlock(hidden_size, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.final_ln = nn.LayerNorm(hidden_size, eps=1e-12)
        self.sid_heads = nn.ModuleList([
            nn.Linear(hidden_size, vocab) for vocab in sid_vocab_sizes
        ])
        self.apply(self._init_weights)
        nn.init.zeros_(self.rel_attn_bias.pos_bias.weight)
        nn.init.zeros_(self.rel_attn_bias.time_bias.weight)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    @staticmethod
    def _causal_mask(seq_len, device):
        return torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1
        )

    def encode(self, batch):
        device = batch['sid'].device
        seq_len = batch['sid'].size(1)
        x = sum(
            embedding(batch['sid'][:, :, level])
            for level, embedding in enumerate(self.sid_embeddings)
        )
        x = x + self.behavior_emb(batch['behavior']) + self.time_gap_emb(batch['time_gap'])
        x = self.emb_dropout(x)
        rel_bias = self.rel_attn_bias(seq_len, batch['time_gap'], device)
        causal_mask = self._causal_mask(seq_len, device)
        for block in self.blocks:
            x = block(x, causal_mask, batch['key_padding_mask'], rel_bias)
        return self.final_ln(x)

    def teacher_forcing_logits(self, hidden, target_sid):
        """Predict c0, then condition c1/c2/collision on preceding gold codes."""
        state = hidden
        logits = []
        for level, head in enumerate(self.sid_heads):
            logits.append(head(state))
            if level < len(self.sid_heads) - 1:
                state = state + self.sid_embeddings[level](target_sid[:, :, level])
        return logits

    def next_code_logits(self, user_hidden, prefix):
        """Return logits for the next SID token conditioned on a generated prefix."""
        level = len(prefix)
        if level >= len(self.sid_heads):
            raise ValueError('Semantic-ID prefix is already complete.')
        state = user_hidden
        for prefix_level, code in enumerate(prefix):
            code_tensor = torch.tensor(code, dtype=torch.long, device=state.device)
            state = state + self.sid_embeddings[prefix_level](code_tensor)
        return self.sid_heads[level](state)

    def forward(self, batch):
        hidden = self.encode(batch)
        logits = self.teacher_forcing_logits(hidden, batch['target_sid'])
        return logits, hidden


# ================================================================
# 3. Multi-level Semantic-ID training
# ================================================================
def train_generative_epoch(model, dataloader, optimizer, scaler, scheduler, device):
    model.train()
    losses = []
    criteria = [
        nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.05)
        for _ in sid_vocab_sizes
    ]
    use_amp = device.type == 'cuda'

    for step, batch in enumerate(dataloader):
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits, _ = model(batch)
            target_sid = batch['target_sid']
            loss = sum(
                criteria[level](
                    logits[level].reshape(-1, sid_vocab_sizes[level]),
                    target_sid[:, :, level].reshape(-1),
                )
                for level in range(len(sid_vocab_sizes))
            ) / len(sid_vocab_sizes)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        losses.append(float(loss.item()))

        if step % 100 == 0:
            print(f'Step {step:04d} | Semantic-ID CE: {loss.item():.4f}')

    return float(np.mean(losses)) if losses else 0.0


# ================================================================
# 4. Training entry point
# ================================================================
MAX_SEQ_LEN = 100
BATCH_SIZE = 8
EPOCHS = 2

train_dataset = GenerativeRecDataset(
    '/kaggle/working/processed_train/chunk_0.parquet',
    sid_table=sid_table,
    max_seq_len=MAX_SEQ_LEN,
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=False,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = GenerativeHSTU(
    sid_vocab_sizes=sid_vocab_sizes,
    behavior_vocab=5,
    max_seq_len=MAX_SEQ_LEN,
    hidden_size=48,
    num_heads=4,
    num_layers=1,
    dropout=0.2,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=1e-4)
total_steps = max(1, EPOCHS * len(train_loader))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-5
)
try:
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

for epoch in range(EPOCHS):
    print(f'\n========== Epoch {epoch + 1}/{EPOCHS} ==========')
    epoch_loss = train_generative_epoch(
        model, train_loader, optimizer, scaler, scheduler, device
    )
    print(f'Epoch {epoch + 1} | average loss: {epoch_loss:.4f}')

torch.save(model.state_dict(), '/kaggle/working/generative_hstu_semantic_id.pt')
print('Saved: /kaggle/working/generative_hstu_semantic_id.pt')

In [ ]:
import math

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


def decode_sid_candidates(model, user_hidden, sid_to_items, sid_prefix_next,
                          seen_items=None, beam_size=50, branch_topk=32, top_k=50):
    """Prefix-constrained, token-level autoregressive SID beam search."""
    seen_items = set(seen_items or [])
    beams = [(tuple(), 0.0)]

    for _ in range(len(model.sid_heads)):
        next_beams = []
        for prefix, prefix_score in beams:
            allowed_codes = sorted(sid_prefix_next.get(prefix, []))
            if not allowed_codes:
                continue
            logits = model.next_code_logits(user_hidden, prefix).float()
            log_probs = F.log_softmax(logits, dim=-1)
            allowed = torch.tensor(allowed_codes, dtype=torch.long, device=logits.device)
            allowed_scores = log_probs.index_select(0, allowed)
            k = min(branch_topk, allowed_scores.numel())
            values, positions = torch.topk(allowed_scores, k=k)
            for value, position in zip(values.tolist(), positions.tolist()):
                code = int(allowed_codes[position])
                next_beams.append((prefix + (code,), prefix_score + float(value)))
        next_beams.sort(key=lambda pair: pair[1], reverse=True)
        beams = next_beams[:beam_size]

    item_scores = {}
    for sid, score in beams:
        for item_id in sid_to_items.get(sid, []):
            if item_id not in seen_items:
                item_scores[item_id] = max(item_scores.get(item_id, -float('inf')), score)
    return [
        item_id for item_id, _ in sorted(
            item_scores.items(), key=lambda pair: pair[1], reverse=True
        )[:top_k]
    ]


def _row_to_batch(user_row, sid_table, max_seq_len, device):
    items = np.asarray(user_row['item_seq'], dtype=np.int64)
    behavior = np.asarray(user_row['behavior_seq'], dtype=np.int64)
    time_gap = np.clip(np.asarray(user_row['time_gap_seq'], dtype=np.int64), 0, 100)
    valid = items > 0
    items, behavior, time_gap = items[valid], behavior[valid], time_gap[valid]
    items = items[-max_seq_len:]
    behavior = behavior[-max_seq_len:]
    time_gap = time_gap[-max_seq_len:]
    if len(items) == 0:
        items = np.array([0], dtype=np.int64)
        behavior = np.array([0], dtype=np.int64)
        time_gap = np.array([0], dtype=np.int64)

    sid = np.zeros((len(items), sid_table.shape[1]), dtype=np.int64)
    in_vocab = (items > 0) & (items < len(sid_table))
    sid[in_vocab] = sid_table[items[in_vocab]]
    seq_len = len(items)

    def pad_1d(values):
        out = np.zeros(max_seq_len, dtype=np.int64)
        out[:seq_len] = values
        return torch.from_numpy(out)

    def pad_2d(values):
        out = np.zeros((max_seq_len, sid_table.shape[1]), dtype=np.int64)
        out[:seq_len] = values
        return torch.from_numpy(out)

    batch = {
        'sid': pad_2d(sid).unsqueeze(0),
        'item': pad_1d(items).unsqueeze(0),
        'behavior': pad_1d(behavior).unsqueeze(0),
        'time_gap': pad_1d(time_gap).unsqueeze(0),
        'key_padding_mask': torch.cat([
            torch.zeros(seq_len, dtype=torch.bool),
            torch.ones(max_seq_len - seq_len, dtype=torch.bool),
        ]).unsqueeze(0),
    }
    return {key: value.to(device) for key, value in batch.items()}


@torch.no_grad()
def generate_for_user(user_row, model, device, sid_table, sid_to_items,
                      sid_prefix_next, max_seq_len=200, top_k=50):
    model.eval()
    batch = _row_to_batch(user_row, sid_table, max_seq_len, device)
    hidden = model.encode(batch)
    last_index = int((~batch['key_padding_mask'][0]).sum().item()) - 1
    user_hidden = hidden[0, last_index]
    seen = set(batch['item'][0, :last_index + 1].detach().cpu().tolist())
    seen.discard(0)
    return decode_sid_candidates(
        model, user_hidden, sid_to_items, sid_prefix_next,
        seen_items=seen, beam_size=50, branch_topk=32, top_k=top_k,
    )


def generative_serving_pipeline(user_row, model, device, top_k=50):
    return generate_for_user(
        user_row, model, device, sid_table, sid_to_items, sid_prefix_next,
        max_seq_len=MAX_SEQ_LEN, top_k=top_k,
    )


@torch.no_grad()
def evaluate_generative_recommender(model, dataloader, device, top_k_list=(10, 50)):
    model.eval()
    hit = {k: 0.0 for k in top_k_list}
    ndcg = {k: 0.0 for k in top_k_list}
    total = 0

    for batch in tqdm(dataloader, desc='Evaluating RQ-VAE generative recommender'):
        batch_device = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }
        hidden = model.encode(batch_device)
        valid_lens = (~batch_device['key_padding_mask']).sum(dim=1)

        for row_idx in range(batch_device['sid'].size(0)):
            last_index = int(valid_lens[row_idx].item()) - 1
            if last_index < 0:
                continue
            target = int(batch_device['target_item'][row_idx, last_index].item())
            if target <= 0:
                continue

            user_hidden = hidden[row_idx, last_index]
            seen = set(batch_device['item'][row_idx, :last_index + 1].cpu().tolist())
            seen.discard(0)
            predictions = decode_sid_candidates(
                model, user_hidden, sid_to_items, sid_prefix_next,
                seen_items=seen, beam_size=50, branch_topk=32,
                top_k=max(top_k_list),
            )
            total += 1
            if target not in predictions:
                continue
            rank = predictions.index(target) + 1
            for k in top_k_list:
                if rank <= k:
                    hit[k] += 1.0
                    ndcg[k] += 1.0 / math.log2(rank + 1)

    normalized = {}
    for k in top_k_list:
        normalized[f'HR@{k}'] = hit[k] / total if total else 0.0
        normalized[f'NDCG@{k}'] = ndcg[k] / total if total else 0.0

    print('\n' + '=' * 64)
    print('Collaborative RQ-VAE Generative Recommender Evaluation')
    print('-' * 64)
    for k in top_k_list:
        print(f'Top-{k:02d} | HR: {normalized[f"HR@{k}"]:.4f} | '
              f'NDCG: {normalized[f"NDCG@{k}"]:.4f}')
    print(f'Total test samples: {total}')
    print('=' * 64)
    return {'normalized': normalized, 'total_samples': total}

In [ ]:
print('\n' + '🔥' * 20)
print('Evaluating collaborative RQ-VAE Semantic-ID recommender...')

test_dataset = GenerativeRecDataset(
    '/kaggle/working/processed_test/chunk_0.parquet',
    sid_table=sid_table,
    max_seq_len=MAX_SEQ_LEN,
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False,
    num_workers=0, pin_memory=False,
)

test_metrics = evaluate_generative_recommender(
    model, test_loader, device, top_k_list=(10, 50)
)

In [ ]:
from collections import Counter


# Fill these with the original ItemCF / ItemCF+HSTU results for the ablation table.
baseline_metrics = {
    'HR@10': None,
    'HR@50': None,
    'NDCG@10': None,
    'NDCG@50': None,
}
improved_metrics = test_metrics.get('normalized', {})


def _fmt(value):
    return f'{value:.4f}' if isinstance(value, (float, int)) else 'N/A'


print('\n' + '=' * 70)
print('Ablation: ItemCF / HSTU / metadata SID / collaborative RQ-VAE SID')
print('-' * 70)
print(f"{'Metric':<12} | {'Baseline':>10} | {'Generative':>10} | {'Uplift':>10}")
for key in ['HR@10', 'HR@50', 'NDCG@10', 'NDCG@50']:
    base = baseline_metrics.get(key)
    ours = improved_metrics.get(key)
    uplift = (ours - base) / base if isinstance(base, (float, int)) and base else None
    uplift_text = f'{uplift:+.2%}' if uplift is not None else 'N/A'
    print(f'{key:<12} | {_fmt(base):>10} | {_fmt(ours):>10} | {uplift_text:>10}')
print(f"Samples      | {'-':>10} | {test_metrics.get('total_samples', 'N/A'):>10} | {'-':>10}")
print('=' * 70)


def evaluate_generated_list_quality(sample_users=3000, top_k=50):
    """Coverage and intra-list category diversity of generated recommendations."""
    eval_df = pl.read_parquet('/kaggle/working/processed_test/chunk_0.parquet')
    if len(eval_df) > sample_users:
        eval_df = eval_df.sample(n=sample_users, seed=42)

    unique_items = set()
    valid = 0
    nonempty = 0
    category_diversity = []

    for row in eval_df.iter_rows(named=True):
        recommendations = generative_serving_pipeline(row, model, device, top_k=top_k)
        if not recommendations:
            continue
        nonempty += 1
        unique_items.update(recommendations)
        target = next((x for x in reversed(row['item_seq']) if x > 0), 0)
        if target > 0:
            valid += 1
        categories = [item_to_cat.get(item) for item in recommendations]
        categories = [category for category in categories if category is not None]
        if categories:
            category_diversity.append(len(set(categories)) / len(categories))

    return {
        'SampleUsers': len(eval_df),
        'NonEmptyRate': nonempty / max(1, len(eval_df)),
        'Coverage': len(unique_items) / max(1, len(item_to_cat)),
        'IntraListCategoryDiversity': float(np.mean(category_diversity)) if category_diversity else 0.0,
        'ValidTargets': valid,
    }


generated_quality = evaluate_generated_list_quality(sample_users=3000, top_k=50)
print('\nGenerated list quality:')
for key, value in generated_quality.items():
    print(f'{key:<28}: {value:.6f}' if isinstance(value, float) else f'{key:<28}: {value}')

# 协同信号 + RQ-VAE + HSTU 生成式推荐

本版本不会覆盖上一版 `tmall-generative-recommender-hstu.ipynb`。核心链路为：

```text
Tmall 用户行为序列
    → 行为/时间/活跃度加权的稀疏 item-item 共现样本
    → Weighted Item2Vec / 负采样学习 collaborative item embedding
    → RQ-VAE 残差量化为 3 级语义 code
    → 追加 collision token，保证最终 SID 唯一
    → HSTU 编码历史 SID 与行为/时间上下文
    → teacher forcing 训练逐 token SID decoder
    → 前缀约束 Beam Search 生成合法 SID
    → 映射为 Top-K 商品
```

## 与上一版的关键差异

上一版 SID 使用 `[大类目, 小类目, block, offset]` 的确定性层级编码。本版不使用类目来生成 SID，而是从用户行为共现学习 item embedding，再由 RQ-VAE 学习离散 code；类目只用于最终多样性统计。

本版的 SID decoder 也从“四个 head 独立预测”升级为逐级条件生成：预测 `c0` 后将其 embedding 加入状态，再预测 `c1`、`c2` 和 collision token。推理时只扩展真实 SID 前缀树中存在的 token，因此不会产生无法映射为商品的无效完整路径。

## 规模边界

代码不构建 `N_item × N_item` 稠密矩阵，而是限制每个 item 的共现邻居数，并提供 `MAX_TRAIN_PAIRS` 上限。完整 13.3 亿行日志仍需先按用户和时间聚合为 sequence parquet；本 notebook 与原始版本一样，从已经聚合的 `hstu_seq_chunk_00.parquet` 开始。

首次运行建议先使用较小数据分片验证流程，再逐步提高：

- `MAX_TRAIN_PAIRS`
- `ITEM_EMBED_DIM`
- `ITEM2VEC_EPOCHS`
- `RQ_EPOCHS`
- HSTU 的 `hidden_size`、`num_layers` 和序列长度

需要真实跑完 ItemCF、原始 HSTU、metadata SID、collaborative RQ-VAE SID 四组实验后，才能在简历中填写提升比例。